In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Подготовка таблицы

In [2]:
df_pedagog1 = pd.read_excel('./data/Педагоги.xlsx', sheet_name='2022')
df_pedagog1['year']=2022

df_pedagog2 = pd.read_excel('./data/Педагоги.xlsx', sheet_name='2023')
df_pedagog2['year']=2023

df_pedagog3 = pd.read_excel('./data/Педагоги.xlsx', sheet_name='2024')
df_pedagog3['year']=2024

df_pedagog_all = pd.concat([df_pedagog1, df_pedagog2, df_pedagog3], axis=0)

In [3]:
df_pedagog_all = df_pedagog_all.rename(columns={'ID образовательной организации':'school_num',
                                                'Основная должность':'main_position',
                                                'Направление деятельности':'speciality',
                                                'Пол': 'sex',
                                                'Возраст':'age',
                                                'Ученая степень':'academic_degree',
                                                'Квалификационная категория (педагогического работника)':'category_qual',
                                                'Дата присвоения квалификационной категории':'category_date',
                                                'Срок действия квалификационной категории':'category_due',
                                                'Общий стаж работы':'experience_age_all',
                                                'Стаж педагогической работы':'experience_age_pedagogic',
                                                'Предмет':'subject',
                                                'Параллель':'parallel',
                                                'Имеет классное руководство':'class_manager'})

keywords = ['педагог', 'учитель', 'методист', 'преподават', 'воспитатель', 'тьютор']
matching_pedagog = set([item for item in df_pedagog_all.main_position.values
            if any(sub in str(item).lower() for sub in keywords)])
matching_pedagog = set([item for item in matching_pedagog if 'заместитель' not in str(item).lower()])

keywords_manager = ['заместитель', 'ведущий', 'заведующий', 'руководитель', 'начальник', 'главный']
matching_manager = set([item for item in df_pedagog_all.main_position.values
            if any(sub in str(item).lower() for sub in keywords_manager)])

matching_other = set(df_pedagog_all.main_position.unique()).difference(matching_pedagog.union(matching_manager))

In [4]:
matching_dict = {}

for val in matching_pedagog:
    matching_dict[val] = 'учитель'
for val in matching_manager:
    matching_dict[val] = 'руководитель'
for val in matching_other:
    matching_dict[val] = 'другой'

In [5]:
df_pedagog_all['main_position_cat'] = df_pedagog_all['main_position'].map(matching_dict).fillna('не_найдено')

In [6]:
df_pedagog_all['sex'] = df_pedagog_all['sex'].map({'Мужской': 1, 'Женский': 0})

In [7]:
keywords = {'кандидат': 'кандидат', 'доктор': 'доктор'}

def find_academic_degree(text):
    if pd.isna(text):
        return 'не_найдено'

    text = str(text).lower()
    for word, code in keywords.items():
        if word in text:
            return code
    return 'не_найдено'

df_pedagog_all['academic_degree_cat'] = df_pedagog_all['academic_degree'].apply(find_academic_degree)

In [8]:
keywords = {'высшая': 'высшая', 'первая': 'первая', 'вторая': 'вторая'}

def find_category_qual(text):
    if pd.isna(text):
        return 'не_найдено'

    text = str(text).lower()
    for word, code in keywords.items():
        if word in text:
            return code
    return 'не_найдено'

# Применяем функцию
df_pedagog_all['category_qual_cat'] = df_pedagog_all['category_qual'].apply(find_category_qual)

In [9]:
df_pedagog_all['category_date'] = df_pedagog_all['category_date'].astype('Int64')
df_pedagog_all['category_due'] = df_pedagog_all['category_due'].astype('Int64')

In [10]:
df_pedagog_all['category_active'] = df_pedagog_all.apply(lambda row: "не было" if pd.isna(row['category_due']) # Если нет данных
                                    else ("активно" if pd.to_datetime(row['category_due']) > pd.to_datetime(row['year']) # Если текущий год меньше года действия
                                    else "не активно"), axis=1)

In [11]:
df_pedagog_all['experience_age_other'] = df_pedagog_all.experience_age_all - df_pedagog_all.experience_age_pedagogic

In [12]:
df_pedagog_all['n_unique_subjects'] = (
    df_pedagog_all.groupby(['year', 'id'])['subject'].transform('nunique')
)

In [13]:
df_pedagog_all['min_parallel'] = (
    df_pedagog_all.groupby(['year', 'id'])['parallel'].transform('min')
)

In [14]:
df_pedagog_all['max_parallel'] = (
    df_pedagog_all.groupby(['year', 'id'])['parallel'].transform('max')
)

In [15]:
df_pedagog_all['n_unique_parallel'] = (
    df_pedagog_all.groupby(['year', 'id'])['parallel'].transform('nunique')
)

In [16]:
df_pedagog_all['class_manager'] = df_pedagog_all['class_manager'].map({'Да': 1, 'Нет': 0})

In [17]:
df_pedagog_all = df_pedagog_all.sort_values(['id', 'year'])
df_pedagog_all['max_year'] = df_pedagog_all.groupby('id')['year'].transform('max')
df_pedagog_all['is_out_next_year'] = (df_pedagog_all['year'] == df_pedagog_all['max_year']).astype(int)
df_pedagog_all = df_pedagog_all.drop(columns=['max_year'])

In [18]:
RULES = {
    'учитель': [
        'английского языка', 'русского языка', 'французского языка',
        'немецкого языка', 'испанского языка', 'итальянского языка',
        'китайского языка', 'арабского языка', 'родного языка',
        'родной литературы', 'литературы', 'литературного чтения',
        'истории', 'всеобщей истории', 'истории России',
        'обществознания', 'обществознания и естествознания (окружающий мир)',
        'права', 'экономики',
        'основ духовно-нравственной культуры народов России',
        'математики', 'алгебры', 'геометрии', 'физики', 'информатики',
        'математики и информатики', 'в сфере ИКТ', 'естествознания',
        'биологии', 'химии', 'географии',
        'изобразительного искусства', 'музыки', 'технологии',
        'физической культуры', 'основ безопасности жизнедеятельности',
        'начальных классов',
    ],

    'воспитатель': [
        'офицер кадетского класса', 'офицер-воспитатель кадетского класса',
        'по социальной работе',
        'по социализации, воспитанию и безопасности обучающихся',
        'по профориентации',
        'группы полного дня, группы ОВЗ', 'группы полного дня',
        'группы продленного дня', 'группы кратковременного пребывания',
        'группы ОВЗ', 'интерната, группы ОВЗ',
    ],

    'управление': [
        'по контролю качества образования', 'учебной части',
        'по содержанию образования, конвергенции образовательных программ',
        'по комплексной безопасности',
        'по материально-техническому обеспечению',
        'по управлению ресурсами', 'по питанию',
        'по финансово-экономической деятельности', 'по учету контингента',
        'по музейной работе', 'учебно-производственной мастерской',
    ],
}

In [19]:
value_to_group = {}
for group, values in RULES.items():
    for v in values:
        value_to_group[v.strip().lower()] = group


def categorize(value):
    if pd.isna(value):
        return 'не указано'
    v = str(value).strip().lower()
    return value_to_group.get(v, 'прочее')

In [20]:
df_pedagog_all['speciality_cat'] = df_pedagog_all['speciality'].apply(categorize)

In [21]:
df_pedagog_clean = df_pedagog_all.drop_duplicates(subset=['id', 'year'], keep='first')

In [22]:
del df_pedagog_clean['academic_degree']
del df_pedagog_clean['main_position']
del df_pedagog_clean['speciality']
del df_pedagog_clean['category_qual']
del df_pedagog_clean['subject']
del df_pedagog_clean['parallel']

In [23]:
del df_pedagog_clean['age']
del df_pedagog_clean['experience_age_all']
del df_pedagog_clean['category_due']

df_pedagog_clean = df_pedagog_clean[df_pedagog_clean['year'] != 2024]

In [24]:
from scipy.stats.mstats import winsorize

df_pedagog_clean['experience_age_pedagogic'] = np.array(
    winsorize(df_pedagog_clean['experience_age_pedagogic'].values, limits=[0.01, 0.01])
)

In [25]:
df_pedagog_clean['n_unique_subjects'] = np.log1p(df_pedagog_clean['n_unique_subjects'])
df_pedagog_clean.loc[df_pedagog_clean['experience_age_other'] < 0, 'experience_age_other'] = 0
df_pedagog_clean['category_date'] = df_pedagog_clean['category_date'].fillna(1900)

In [26]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler,
    RobustScaler, FunctionTransformer, TargetEncoder,
)

In [27]:
def _academic_degree_bin(X):
    mapping = {"не_найдено": 0, "кандидат": 1, "доктор": 1}
    out = X["academic_degree_cat"].map(mapping).fillna(0).astype(int)
    return out.to_frame("academic_degree_bin")

def _category_qual_grouped(X):
    mapping = {"не_найдено": 0, "первая": 1, "вторая": 1, "высшая": 2}
    out = X["category_qual_cat"].map(mapping).fillna(0).astype(int)
    return out.to_frame("category_qual_grouped")

academic_degree_step = FunctionTransformer(
    _academic_degree_bin, validate=False,
    feature_names_out=lambda self, input_features: np.array(["academic_degree_bin"]),
)

category_qual_grouped_step = FunctionTransformer(
    _category_qual_grouped, validate=False,
    feature_names_out=lambda self, input_features: np.array(["category_qual_grouped"]),
)

In [28]:
# где требуется кодирование и масштабирование

preprocess_scaled = ColumnTransformer([
    # target encoding
    ('te', TargetEncoder(target_type='binary', smooth='auto', cv=5, random_state=42),
     ['school_num']),

    # standard
    ('std', StandardScaler(),
     ['n_unique_subjects', 'n_unique_parallel', 'experience_age_pedagogic']),

    # minmax
    ('minmax', MinMaxScaler(), ['min_parallel', 'max_parallel']),

    # robust
    ('robust', RobustScaler(), ['experience_age_other']),

    # бинарные как есть
    ('bin', 'passthrough', ['sex', 'class_manager']),

    # бинарка
    ('academic', academic_degree_step, ['academic_degree_cat']),

    # map + standard
    ('qual', Pipeline([
        ('map', category_qual_grouped_step),
        ('scaler', StandardScaler()),
    ]), ['category_qual_cat']),

    # ordinal + minmax
    ('active', Pipeline([
        ('ord', OrdinalEncoder(
            categories=[['не было', 'не активно', 'активно']],
            handle_unknown='use_encoded_value', unknown_value=-1,
        )),
        ('scaler', MinMaxScaler()),
    ]), ['category_active']),

    # OHE
    ('ohe_position', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
     ['main_position_cat']),
    ('ohe_speciality', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
     ['speciality_cat']),
], remainder='drop', verbose_feature_names_out=False)

In [29]:
# где требуется только кодирование

preprocess_tree = ColumnTransformer([
    ('te', TargetEncoder(target_type='binary', smooth='auto', cv=5, random_state=42),
     ['school_num']),

    ('num', 'passthrough',
     ['sex', 'class_manager', 'experience_age_pedagogic', 'experience_age_other',
      'n_unique_subjects', 'min_parallel', 'max_parallel', 'n_unique_parallel']),

    ('academic', academic_degree_step, ['academic_degree_cat']),

    ('qual', category_qual_grouped_step, ['category_qual_cat']),

    ('active', OrdinalEncoder(
        categories=[['не было', 'не активно', 'активно']],
        handle_unknown='use_encoded_value', unknown_value=-1,
    ), ['category_active']),

    ('ohe_position', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
     ['main_position_cat']),
    ('ohe_speciality', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
     ['speciality_cat']),
], remainder='drop', verbose_feature_names_out=False)

In [30]:
FEATURES = [
    'school_num', 'sex', 'experience_age_pedagogic', 'class_manager',
    'main_position_cat', 'academic_degree_cat', 'category_qual_cat',
    'category_active', 'experience_age_other', 'n_unique_subjects',
    'min_parallel', 'max_parallel', 'n_unique_parallel', 'speciality_cat',
]
# исключили 'category_date', 'id', 'year', 'is_out_next_year'

train = df_pedagog_clean[df_pedagog_clean['year'] == 2022]
test  = df_pedagog_clean[df_pedagog_clean['year'] == 2023]

X_train, y_train = train[FEATURES], train['is_out_next_year']
X_test,  y_test  = test[FEATURES],  test['is_out_next_year']

# Обогощение со школами

In [31]:
print("=== df_pedagog_clean ===")
print(f"Shape: {df_pedagog_clean.shape}")
print(f"Годы: {sorted(df_pedagog_clean['year'].unique())}")
print(f"\nСтрок по годам:")
print(df_pedagog_clean['year'].value_counts().sort_index())
print(f"\nУникальных id: {df_pedagog_clean['id'].nunique()}")
print(f"Уникальных school_num: {df_pedagog_clean['school_num'].nunique()}")

print(f"\n=== Колонки ===")
for c in df_pedagog_clean.columns:
    print(f"  {c:35s} {str(df_pedagog_clean[c].dtype):10s} na={df_pedagog_clean[c].isna().sum()}")

print(f"\n=== Таргет по годам ===")
print(df_pedagog_clean.groupby('year')['is_out_next_year'].agg(['size','sum','mean']))

=== df_pedagog_clean ===
Shape: (35632, 18)
Годы: [np.int64(2022), np.int64(2023)]

Строк по годам:
year
2022    17833
2023    17799
Name: count, dtype: int64

Уникальных id: 20268
Уникальных school_num: 459

=== Колонки ===
  id                                  object     na=0
  school_num                          int64      na=0
  sex                                 int64      na=0
  category_date                       Int64      na=0
  experience_age_pedagogic            int64      na=0
  class_manager                       int64      na=0
  year                                int64      na=0
  main_position_cat                   object     na=0
  academic_degree_cat                 object     na=0
  category_qual_cat                   object     na=0
  category_active                     object     na=0
  experience_age_other                int64      na=0
  n_unique_subjects                   float64    na=0
  min_parallel                        int64      na=0
  max_parallel     

Изучение школ

In [32]:
df_school = pd.read_excel('./data/Школы.xlsx')
df_school = df_school.rename(columns={
    'Школа': 'school_num',
    'Округ': 'district',
    'Кол-во зданий': 'buildings_num',
    'Общая численность контингента': 'students_num',
})

In [33]:
print(f"Shape: {df_school.shape}")
print(f"Колонки: {list(df_school.columns)}")
print(f"Типы:\n{df_school.dtypes}")
print(f"Дубли school_num: {df_school['school_num'].duplicated().sum()}")
print(f"Пропуски:\n{df_school.isna().sum()}")
print(df_school.head())

Shape: (266, 4)
Колонки: ['school_num', 'district', 'buildings_num', 'students_num']
Типы:
school_num        int64
district         object
buildings_num     int64
students_num      int64
dtype: object
Дубли school_num: 0
Пропуски:
school_num       0
district         0
buildings_num    0
students_num     0
dtype: int64
   school_num             district  buildings_num  students_num
0         325  Инновационный округ              6          1294
1         397  Инновационный округ              3           842
2         192       Северный округ             13          4505
3         385       Западный округ             12          3617
4         269      Восточный округ             15          4262


In [34]:
ref_schools = set(df_school['school_num'].unique())

n_before = len(df_pedagog_clean)
df_pedagog_clean = df_pedagog_clean[df_pedagog_clean['school_num'].isin(ref_schools)].copy()
n_after = len(df_pedagog_clean)
print(f"Было {n_before} → стало {n_after} (отрезали {n_before - n_after})")

df_pedagog_clean = df_pedagog_clean.merge(
    df_school[['school_num', 'district', 'buildings_num', 'students_num']],
    on='school_num', how='left', validate='many_to_one',
)

print(f"Пропуски: {df_pedagog_clean[['district','buildings_num','students_num']].isna().sum().to_dict()}")
print(f"Shape: {df_pedagog_clean.shape}")
print(f"\nТаргет по годам после фильтра:")
print(df_pedagog_clean.groupby('year')['is_out_next_year'].agg(['size','sum','mean']))

Было 35632 → стало 34789 (отрезали 843)
Пропуски: {'district': 0, 'buildings_num': 0, 'students_num': 0}
Shape: (34789, 21)

Таргет по годам после фильтра:
       size   sum      mean
year                       
2022  17395  1983  0.113998
2023  17394  2095  0.120444


Учителя и школы. Берём только с 2022, чтобы не было утечки данных

In [35]:
# 1. Размер школы в 2022 (по train)
school_size_2022 = (
    df_pedagog_clean[df_pedagog_clean['year'] == 2022]
    .groupby('school_num').size()
    .rename('school_size')
)

# 2. Мержим во все строки (значение из 2022 для обеих)
df_pedagog_clean = df_pedagog_clean.merge(
    school_size_2022, on='school_num', how='left', validate='many_to_one',
)

n_missing = df_pedagog_clean['school_size'].isna().sum()
print(f"Пропусков school_size до fillna: {n_missing}")

# 3. Медиана students/teacher в 2022
tmp_2022 = df_pedagog_clean[df_pedagog_clean['year'] == 2022]
sp_median = (tmp_2022['students_num'] / tmp_2022['school_size']).median()
print(f"Медиана students_per_teacher 2022: {sp_median:.1f}")

# 4. Заполняем 6 школ через students_num / sp_median
mask_missing = df_pedagog_clean['school_size'].isna()
df_pedagog_clean.loc[mask_missing, 'school_size'] = (
    df_pedagog_clean.loc[mask_missing, 'students_num'] / sp_median
).round()

# 5. students_per_teacher как фича
df_pedagog_clean['students_per_teacher'] = (
    df_pedagog_clean['students_num'] / df_pedagog_clean['school_size']
)

# Проверки
print(f"Пропусков после fillna: {df_pedagog_clean['school_size'].isna().sum()}")
print("\n=== school_size по годам ===")
print(df_pedagog_clean.groupby('year')['school_size'].describe())
print("\n=== students_per_teacher ===")
print(df_pedagog_clean['students_per_teacher'].describe())
print("\n=== 6 заполненных школ ===")
print(df_pedagog_clean[df_pedagog_clean['school_num'].isin([462,199,261,477,28,354])]
      .groupby('school_num')[['school_size','students_num','students_per_teacher']].first())

Пропусков school_size до fillna: 9
Медиана students_per_teacher 2022: 23.7
Пропусков после fillna: 0

=== school_size по годам ===
        count        mean        std  min   25%    50%    75%    max
year                                                                
2022  17395.0  105.774130  39.215207  1.0  80.0  106.0  134.0  181.0
2023  17394.0  105.833736  39.303081  1.0  80.0  107.0  134.0  225.0

=== students_per_teacher ===
count    34789.000000
mean        43.665057
std        199.974868
min          4.118644
25%         20.734513
50%         23.732026
75%         26.356061
max       5687.000000
Name: students_per_teacher, dtype: float64

=== 6 заполненных школ ===
            school_size  students_num  students_per_teacher
school_num                                                 
28                211.0          5009             23.739336
199               170.0          4027             23.688235
261               220.0          5221             23.731818
354             

In [36]:
# Топ-10 по students_per_teacher
top_sp = df_pedagog_clean.nlargest(10, 'students_per_teacher')[
    ['school_num', 'students_num', 'school_size', 'students_per_teacher']
].drop_duplicates('school_num')
print(top_sp)

       school_num  students_num  school_size  students_per_teacher
4722           82          5687          1.0                5687.0
32298         198          5535          1.0                5535.0


In [37]:
# Сколько школ с school_size <= 2 и students_num > 2000
anomalies = (
    df_pedagog_clean.groupby('school_num')
    .agg(school_size=('school_size', 'first'),
         students_num=('students_num', 'first'))
)
anomalies['sp'] = anomalies['students_num'] / anomalies['school_size']
print(f"Школ с school_size <= 2 и students_num > 2000: "
      f"{((anomalies['school_size'] <= 2) & (anomalies['students_num'] > 2000)).sum()}")
print(anomalies[(anomalies['school_size'] <= 2) & (anomalies['students_num'] > 2000)])

Школ с school_size <= 2 и students_num > 2000: 17
            school_size  students_num      sp
school_num                                   
67                  1.0          4139  4139.0
82                  1.0          5687  5687.0
113                 2.0          2064  1032.0
148                 2.0          5028  2514.0
168                 2.0          4853  2426.5
192                 1.0          4505  4505.0
198                 1.0          5535  5535.0
200                 2.0          3634  1817.0
201                 2.0          7124  3562.0
225                 1.0          2017  2017.0
270                 2.0          3721  1860.5
272                 1.0          3758  3758.0
334                 1.0          4574  4574.0
347                 2.0          5122  2561.0
429                 1.0          2129  2129.0
440                 1.0          2688  2688.0
483                 2.0          4978  2489.0


In [38]:
df_pedagog_clean.groupby('school_num')['school_size'].first().describe()

,school_size
count,266.000000
mean,69.131579
std,53.540899
min,1.000000
25%,9.000000
50%,66.000000
75%,108.750000
max,225.000000


In [39]:
df_pedagog_clean.nlargest(10, 'students_per_teacher')[
    ['school_num','students_num','school_size','students_per_teacher']
].drop_duplicates('school_num')

,school_num,students_num,school_size,students_per_teacher
4722,82,5687,1.0,5687.0
32298,198,5535,1.0,5535.0


In [40]:
anom = df_pedagog_clean.groupby('school_num').agg(
    school_size=('school_size','first'),
    students_num=('students_num','first'),
)
anom['sp'] = anom['students_num']/anom['school_size']
print(f"school_size <= 2 и students_num > 2000: {((anom['school_size']<=2)&(anom['students_num']>2000)).sum()}")

school_size <= 2 и students_num > 2000: 17


In [41]:
anom_schools = [67, 82, 113, 148, 168, 192, 198, 200, 201, 225, 270, 272, 334, 347, 429, 440, 483]

anom_df = df_pedagog_clean[df_pedagog_clean['school_num'].isin(anom_schools)]
print(f"Строк в 17 школах: {len(anom_df)} ({len(anom_df)/len(df_pedagog_clean)*100:.2f}%)")
print(f"Уникальных учителей: {anom_df['id'].nunique()}")
print(f"\nТаргет по годам в этих школах:")
print(anom_df.groupby('year')['is_out_next_year'].agg(['size','sum','mean']))
print(f"\nТаргет по годам во всех остальных:")
print(df_pedagog_clean[~df_pedagog_clean['school_num'].isin(anom_schools)]
      .groupby('year')['is_out_next_year'].agg(['size','sum','mean']))

Строк в 17 школах: 82 (0.24%)
Уникальных учителей: 77

Таргет по годам в этих школах:
      size  sum      mean
year                     
2022    25    0  0.000000
2023    57    1  0.017544

Таргет по годам во всех остальных:
       size   sum      mean
year                       
2022  17370  1983  0.114162
2023  17337  2094  0.120782


In [42]:
anom_ids = df_pedagog_clean[df_pedagog_clean['school_num'].isin(anom_schools)]['id'].unique()
anom_full = df_pedagog_clean[df_pedagog_clean['id'].isin(anom_ids)].sort_values(['id','year'])
print(f"Учителей в 17 школах (уник. id): {len(anom_ids)}")
print(f"\nГде они были в 2023:")
# для каждого учителя из 2022 в аномальной школе — куда он попал в 2023
anom_2022 = df_pedagog_clean[(df_pedagog_clean['year']==2022) & (df_pedagog_clean['school_num'].isin(anom_schools))][['id','school_num']]
anom_2023 = df_pedagog_clean[df_pedagog_clean['year']==2023][['id','school_num']].rename(columns={'school_num':'school_2023'})
merged = anom_2022.merge(anom_2023, on='id', how='left')
print(f"  - есть в 2023: {merged['school_2023'].notna().sum()}")
print(f"  - нет в 2023: {merged['school_2023'].isna().sum()}")
print(f"  - переехали в другую школу: {(merged['school_2023'].notna() & (merged['school_2023'] != merged['school_num'])).sum()}")

Учителей в 17 школах (уник. id): 77

Где они были в 2023:
  - есть в 2023: 15
  - нет в 2023: 10
  - переехали в другую школу: 10


In [43]:
ids_2022 = set(df_pedagog_clean[df_pedagog_clean['year']==2022]['id'])
ids_2023 = set(df_pedagog_clean[df_pedagog_clean['year']==2023]['id'])
only_2022 = ids_2022 - ids_2023

print(f"Учителей только в 2022 (нет в 2023): {len(only_2022)}")
only_2022_df = df_pedagog_clean[(df_pedagog_clean['year']==2022) & (df_pedagog_clean['id'].isin(only_2022))]
print(f"У них таргет: {only_2022_df['is_out_next_year'].value_counts().to_dict()}")

# 2. Для тех, кто есть в 2023, но не в 2024 (не в df_pedagog_clean, т.к. 2024 отрезан) —
#    таргет для 2023-строки должен быть 1
ids_2024 = set(df_pedagog_all[df_pedagog_all['year']==2024]['id'])  # из сырых
only_2023_not_2024 = ids_2023 - ids_2024
print(f"\nУчителей в 2023, но не в 2024: {len(only_2023_not_2024)}")
only_2023_df = df_pedagog_clean[(df_pedagog_clean['year']==2023) & (df_pedagog_clean['id'].isin(only_2023_not_2024))]
print(f"У них таргет: {only_2023_df['is_out_next_year'].value_counts().to_dict()}")

Учителей только в 2022 (нет в 2023): 2595
У них таргет: {1: 1983, 0: 612}

Учителей в 2023, но не в 2024: 2095
У них таргет: {1: 2095}


In [44]:
anom_schools = [67, 82, 113, 148, 168, 192, 198, 200, 201, 225,
                270, 272, 334, 347, 429, 440, 483]

n_before = len(df_pedagog_clean)
df_pedagog_clean = df_pedagog_clean[~df_pedagog_clean['school_num'].isin(anom_schools)].copy()
n_after = len(df_pedagog_clean)

print(f"Отрезали {n_before - n_after} строк (17 школ)")
print(f"Shape: {df_pedagog_clean.shape}")
print(f"\nТаргет по годам после фильтра:")
print(df_pedagog_clean.groupby('year')['is_out_next_year'].agg(['size','sum','mean']))
print(f"\nУникальных school_num: {df_pedagog_clean['school_num'].nunique()}")

Отрезали 82 строк (17 школ)
Shape: (34707, 23)

Таргет по годам после фильтра:
       size   sum      mean
year                       
2022  17370  1983  0.114162
2023  17337  2094  0.120782

Уникальных school_num: 249


In [45]:
# 1. Удаляем старые school_size и students_per_teacher
df_pedagog_clean = df_pedagog_clean.drop(
    columns=['school_size', 'students_per_teacher'], errors='ignore'
)

# 2. Размер школы в 2022 (уже без 17 аномальных)
school_size_2022 = (
    df_pedagog_clean[df_pedagog_clean['year'] == 2022]
    .groupby('school_num').size()
    .rename('school_size')
)
df_pedagog_clean = df_pedagog_clean.merge(
    school_size_2022, on='school_num', how='left', validate='many_to_one',
)

n_missing = df_pedagog_clean['school_size'].isna().sum()
print(f"Пропусков school_size до fillna: {n_missing}")

# 3. Медиана students/teacher в 2022 (уже без аномалий)
tmp_2022 = df_pedagog_clean[df_pedagog_clean['year'] == 2022]
sp_median = (tmp_2022['students_num'] / tmp_2022['school_size']).median()
print(f"Медиана students_per_teacher 2022: {sp_median:.1f}")

# 4. Заполняем пропуски
mask_missing = df_pedagog_clean['school_size'].isna()
df_pedagog_clean.loc[mask_missing, 'school_size'] = (
    df_pedagog_clean.loc[mask_missing, 'students_num'] / sp_median
).round()
print(f"Пропусков после fillna: {df_pedagog_clean['school_size'].isna().sum()}")

# 5. students_per_teacher (не как фича, а для диагностики — потом уберём)
df_pedagog_clean['students_per_teacher'] = (
    df_pedagog_clean['students_num'] / df_pedagog_clean['school_size']
)

# Проверки
print("\n=== school_size по годам (по строкам) ===")
print(df_pedagog_clean.groupby('year')['school_size'].describe())

print("\n=== school_size по школам (не по строкам) ===")
print(df_pedagog_clean.groupby('school_num')['school_size'].first().describe())

print("\n=== students_per_teacher ===")
print(df_pedagog_clean['students_per_teacher'].describe())

print("\n=== Топ-5 students_per_teacher (должно быть ~ нормально) ===")
print(df_pedagog_clean.nlargest(5, 'students_per_teacher')[
    ['school_num','students_num','school_size','students_per_teacher']
].drop_duplicates('school_num'))

Пропусков school_size до fillna: 9
Медиана students_per_teacher 2022: 23.7
Пропусков после fillna: 0

=== school_size по годам (по строкам) ===
        count        mean        std  min   25%    50%    75%    max
year                                                                
2022  17370.0  105.924007  39.043757  1.0  80.0  106.0  134.0  181.0
2023  17337.0  106.176616  38.909276  1.0  80.0  107.0  134.0  225.0

=== school_size по школам (не по строкам) ===
count    249.000000
mean      73.751004
std       52.229046
min        1.000000
25%       27.000000
50%       76.000000
75%      112.000000
max      225.000000
Name: school_size, dtype: float64

=== students_per_teacher ===
count    34707.000000
mean        36.475442
std        118.286494
min          4.118644
25%         20.734513
50%         23.731818
75%         26.356061
max       1983.000000
Name: students_per_teacher, dtype: float64

=== Топ-5 students_per_teacher (должно быть ~ нормально) ===
      school_num  students_n

In [46]:
# Все школы, где students_per_teacher > 100 (явно неполные данные)
anom2 = (
    df_pedagog_clean.groupby('school_num')
    .agg(school_size=('school_size','first'),
         students_num=('students_num','first'),
         n_rows=('id','size'))
)
anom2['sp'] = anom2['students_num'] / anom2['school_size']
anom2 = anom2[anom2['sp'] > 100].sort_values('sp', ascending=False)
print(f"Школ с students_per_teacher > 100: {len(anom2)}")
print(anom2)

Школ с students_per_teacher > 100: 54
            school_size  students_num  n_rows           sp
school_num                                                
15                  3.0          5949       7  1983.000000
343                 4.0          6430       8  1607.500000
369                 4.0          6411      10  1602.750000
222                 1.0          1562       4  1562.000000
407                 3.0          4600      19  1533.333333
437                 3.0          4585       5  1528.333333
405                 3.0          4434       5  1478.000000
211                 3.0          4332       7  1444.000000
134                 3.0          4302       5  1434.000000
232                 1.0          1350       5  1350.000000
281                 4.0          5385      11  1346.250000
335                 3.0          4035       6  1345.000000
245                 5.0          6634       8  1326.800000
346                 3.0          3966       6  1322.000000
53                

In [47]:
school_stats_2022 = (
    df_pedagog_clean[df_pedagog_clean['year'] == 2022]
    .groupby('school_num')
    .agg(school_size=('school_size', 'first'),
         students_num=('students_num', 'first'))
)
school_stats_2022['sp'] = school_stats_2022['students_num'] / school_stats_2022['school_size']

incomplete_mask = school_stats_2022['sp'] > 100
incomplete_schools = school_stats_2022[incomplete_mask].index.tolist()
print(f"Школ с неполными данными: {len(incomplete_schools)}")

# 2. Медиана students_per_teacher по НОРМАЛЬНЫМ школам
normal_sp_median = school_stats_2022.loc[~incomplete_mask, 'sp'].median()
print(f"Медиана students_per_teacher по нормальным школам: {normal_sp_median:.2f}")

# 3. Сохраняем исходный school_size (для сравнения версий потом)
df_pedagog_clean['school_size_raw'] = df_pedagog_clean['school_size']

# 4. Заменяем school_size для неполных школ
fix_values = (
    df_pedagog_clean.loc[df_pedagog_clean['school_num'].isin(incomplete_schools),
                         'students_num'] / normal_sp_median
).round()

df_pedagog_clean.loc[
    df_pedagog_clean['school_num'].isin(incomplete_schools), 'school_size'
] = fix_values

# 5. Пересчитываем students_per_teacher на основе fixed school_size
df_pedagog_clean['students_per_teacher'] = (
    df_pedagog_clean['students_num'] / df_pedagog_clean['school_size']
)

# Проверки
print("\n=== school_size по школам (после фикса) ===")
print(df_pedagog_clean.groupby('school_num')['school_size'].first().describe())

print("\n=== students_per_teacher (после фикса) ===")
print(df_pedagog_clean['students_per_teacher'].describe())

print("\n=== Топ-5 по students_per_teacher (должно быть ≈ 23.7) ===")
print(df_pedagog_clean.nlargest(5, 'students_per_teacher')[
    ['school_num', 'students_num', 'school_size', 'students_per_teacher']
].drop_duplicates('school_num'))

print("\n=== Примеры исправленных школ ===")
print(df_pedagog_clean[df_pedagog_clean['school_num'].isin([15, 343, 369, 222, 407])]
      [['school_num', 'students_num', 'school_size_raw', 'school_size', 'students_per_teacher']]
      .drop_duplicates('school_num'))

Школ с неполными данными: 54
Медиана students_per_teacher по нормальным школам: 23.54

=== school_size по школам (после фикса) ===
count    249.000000
mean     111.425703
std       60.500896
min        8.000000
25%       64.000000
50%       99.000000
75%      148.000000
max      282.000000
Name: school_size, dtype: float64

=== students_per_teacher (после фикса) ===
count    34707.000000
mean        23.091810
std          4.976008
min          4.118644
25%         20.734513
50%         23.685185
75%         26.099010
max         42.071429
Name: students_per_teacher, dtype: float64

=== Топ-5 по students_per_teacher (должно быть ≈ 23.7) ===
      school_num  students_num  school_size  students_per_teacher
1827         456           589         14.0             42.071429

=== Примеры исправленных школ ===
       school_num  students_num  school_size_raw  school_size  \
628           343          6430              4.0        273.0   
1587          407          4600              3.0       

#ПК-ПП

In [48]:
df_pkpp1 = pd.read_excel('./data/ПК-ПП.xlsx', sheet_name='2022')
df_pkpp1['year'] = 2022
df_pkpp2 = pd.read_excel('./data/ПК-ПП.xlsx', sheet_name='2023')
df_pkpp2['year'] = 2023
df_pkpp3 = pd.read_excel('./data/ПК-ПП.xlsx', sheet_name='2024')
df_pkpp3['year'] = 2024

df_pkpp_all = pd.concat([df_pkpp1, df_pkpp2, df_pkpp3], axis=0)

df_pkpp_all = df_pkpp_all.rename(columns={
    'Наименование программы': 'name_prog',
    'Программа профессиональной переподготовки(ПП)/повышения квалификации(ПК)': 'type_prog',
    'Количество часов': 'hours_tot',
    'Дата получения документа/освоения программы': 'date_done',
})

print(f"Shape: {df_pkpp_all.shape}")
print(f"Годы: {sorted(df_pkpp_all['year'].unique())}")
print(f"\nСтрок по годам:")
print(df_pkpp_all['year'].value_counts().sort_index())
print(f"\nКолонки: {df_pkpp_all.columns.tolist()}")
print(f"\nТипы:\n{df_pkpp_all.dtypes}")
print(f"\nПропуски:\n{df_pkpp_all.isna().sum()}")
print(f"\nУникальных id: {df_pkpp_all['id'].nunique()}")
print(df_pkpp_all.head())

Shape: (65733, 6)
Годы: [np.int64(2022), np.int64(2023), np.int64(2024)]

Строк по годам:
year
2022    28417
2023    21581
2024    15735
Name: count, dtype: int64

Колонки: ['id', 'name_prog', 'type_prog', 'hours_tot', 'date_done', 'year']

Типы:
id           object
name_prog    object
type_prog    object
hours_tot     int64
date_done     int64
year          int64
dtype: object

Пропуски:
id           0
name_prog    0
type_prog    0
hours_tot    0
date_done    0
year         0
dtype: int64

Уникальных id: 19061
           id                                          name_prog type_prog  \
0  18ac5986cd  Методические подходы в подготовке к государств...        ПК   
1  18ac5986cd  Организация образовательного процесса в услови...        ПК   
2  18ac5986cd  Подготовка организаторов пунктов проведения эк...        ПК   
3  9b13af5810  Подготовка организаторов пунктов проведения эк...        ПК   
4  1be672d40b  Инструменты объективного оценивания уровня дос...        ПК   

   hours_tot  

In [49]:
# 1. Оставляем только 2022 и 2023 (2024 — будущее относительно таргета 2023-строк)
df_pkpp = df_pkpp_all[df_pkpp_all['year'].isin([2022, 2023])].copy()
print(f"Строк ПК-ПП после фильтра (2022+2023): {len(df_pkpp)}")

# 2. Проверим расхождение date_done и year
mismatch = (df_pkpp['date_done'] != df_pkpp['year']).sum()
print(f"Строк, где date_done != year: {mismatch} ({mismatch/len(df_pkpp)*100:.2f}%)")
if mismatch > 0:
    print(df_pkpp[df_pkpp['date_done'] != df_pkpp['year']].head())

# 3. Флаг "педагогическая программа"
keywords = ['преподавания', 'курса']
pattern = '|'.join(keywords)
df_pkpp['is_pedagog'] = (
    df_pkpp['name_prog'].str.contains(pattern, na=False, case=False).astype(int)
)
print(f"\nis_pedagog=1: {df_pkpp['is_pedagog'].sum()} ({df_pkpp['is_pedagog'].mean()*100:.1f}%)")

# 4. Кумулята для каждого года
def cumulative_pkpp(df_pkpp, target_year):
    sub = df_pkpp[df_pkpp['year'] <= target_year]
    agg = sub.groupby('id').agg(
        pkpp_cum_courses=('id', 'size'),
        pkpp_cum_hours=('hours_tot', 'sum'),
        pkpp_cum_pedagog=('is_pedagog', 'sum'),
    ).reset_index()
    # уникальные типы за все годы <= target_year
    unique_types = sub.groupby('id')['type_prog'].nunique().rename('pkpp_cum_types').reset_index()
    agg = agg.merge(unique_types, on='id', how='left')
    agg['year'] = target_year
    return agg

cum_2022 = cumulative_pkpp(df_pkpp, 2022)
cum_2023 = cumulative_pkpp(df_pkpp, 2023)
pkpp_cum = pd.concat([cum_2022, cum_2023], ignore_index=True)
print(f"\nКумулята: {len(pkpp_cum)} строк, {pkpp_cum['id'].nunique()} учителей")

# 5. Мержим в педагогов
n_before = len(df_pedagog_clean)
df_pedagog_clean = df_pedagog_clean.merge(
    pkpp_cum, on=['id', 'year'], how='left', validate='many_to_one',
)
pkpp_cols = ['pkpp_cum_courses', 'pkpp_cum_hours', 'pkpp_cum_pedagog', 'pkpp_cum_types']
df_pedagog_clean[pkpp_cols] = df_pedagog_clean[pkpp_cols].fillna(0)
print(f"Merge: было {n_before}, стало {len(df_pedagog_clean)}")

# 6. Проверки
print("\n=== ПК-ПП средние по годам ===")
print(df_pedagog_clean.groupby('year')[pkpp_cols].mean())

print("\n=== ПК-ПП describe ===")
print(df_pedagog_clean[pkpp_cols].describe())

print("\n=== Доля учителей без курсов ===")
for col in ['pkpp_cum_courses']:
    print(f"{col} == 0: {(df_pedagog_clean[col] == 0).mean()*100:.1f}%")

# 7. Ключевая проверка: у 2022-строк НЕ должно быть значений из 2023
#    Для этого проверим, что pkpp_cum_courses для 2022 <= courses за 2022 по этому учителю
courses_2022_only = df_pkpp[df_pkpp['year'] == 2022].groupby('id').size()
row_2022 = df_pedagog_clean[df_pedagog_clean['year'] == 2022].set_index('id')['pkpp_cum_courses']
check = pd.concat([row_2022, courses_2022_only.rename('courses_2022')], axis=1).fillna(0)
violations = (check['pkpp_cum_courses'] > check['courses_2022']).sum()
print(f"\nНарушений (2022-строка с курсами из будущего): {violations}")

Строк ПК-ПП после фильтра (2022+2023): 49998
Строк, где date_done != year: 0 (0.00%)

is_pedagog=1: 684 (1.4%)

Кумулята: 33724 строк, 17870 учителей
Merge: было 34707, стало 34707

=== ПК-ПП средние по годам ===
      pkpp_cum_courses  pkpp_cum_hours  pkpp_cum_pedagog  pkpp_cum_types
year                                                                    
2022          1.482211       32.683305          0.010535        0.814968
2023          2.566476       60.729423          0.036627        0.880429

=== ПК-ПП describe ===
       pkpp_cum_courses  pkpp_cum_hours  pkpp_cum_pedagog  pkpp_cum_types
count      34707.000000    34707.000000      34707.000000    34707.000000
mean           2.023828       46.693030          0.023569        0.847668
std            1.586363       46.527961          0.159119        0.363890
min            0.000000        0.000000          0.000000        0.000000
25%            1.000000       16.000000          0.000000        1.000000
50%            2.000000    

# Сборка

In [65]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder, OrdinalEncoder, StandardScaler,
    MinMaxScaler, RobustScaler, FunctionTransformer, TargetEncoder,
)
from sklearn.pipeline import Pipeline

# --- 1. Вспомогательные трансформеры (один раз) ---
def _academic_degree_bin(X):
    mapping = {"не_найдено": 0, "кандидат": 1, "доктор": 1}
    return X["academic_degree_cat"].map(mapping).fillna(0).astype(int).to_frame("academic_degree_bin")

def _category_qual_grouped(X):
    mapping = {"не_найдено": 0, "первая": 1, "вторая": 1, "высшая": 2}
    return X["category_qual_cat"].map(mapping).fillna(0).astype(int).to_frame("category_qual_grouped")

academic_degree_step = FunctionTransformer(
    _academic_degree_bin, validate=False,
    feature_names_out=lambda self, input_features: np.array(["academic_degree_bin"]),
)
category_qual_grouped_step = FunctionTransformer(
    _category_qual_grouped, validate=False,
    feature_names_out=lambda self, input_features: np.array(["category_qual_grouped"]),
)

# --- 2. Наборы признаков ---
base_features = [
    'school_num', 'sex', 'experience_age_pedagogic', 'class_manager',
    'main_position_cat', 'academic_degree_cat', 'category_qual_cat',
    'category_active', 'experience_age_other', 'n_unique_subjects',
    'min_parallel', 'max_parallel', 'n_unique_parallel', 'speciality_cat',
]
school_features = ['district', 'buildings_num', 'students_num', 'school_size']
pkpp_features = ['pkpp_cum_courses', 'pkpp_cum_hours', 'pkpp_cum_types']

FEATURES_1 = base_features
FEATURES_2 = base_features + school_features + pkpp_features + ['students_per_teacher']
FEATURES_3 = base_features + school_features + pkpp_features
FEATURES_4 = base_features + pkpp_features

# --- 3. Препроцессоры под конкретный набор ---
def build_preprocess_tree(features):
    num_passthrough = [f for f in features if f in [
        'sex', 'class_manager', 'experience_age_pedagogic', 'experience_age_other',
        'n_unique_subjects', 'min_parallel', 'max_parallel', 'n_unique_parallel',
        'buildings_num', 'students_num', 'school_size', 'students_per_teacher',
        'pkpp_cum_courses', 'pkpp_cum_hours', 'pkpp_cum_types',
    ]]
    transformers = [
        ('te', TargetEncoder(target_type='binary', smooth='auto', cv=5, random_state=42), ['school_num']),
        ('num', 'passthrough', num_passthrough),
        ('academic', academic_degree_step, ['academic_degree_cat']),
        ('qual', category_qual_grouped_step, ['category_qual_cat']),
        ('active', OrdinalEncoder(categories=[['не было', 'не активно', 'активно']],
                                  handle_unknown='use_encoded_value', unknown_value=-1), ['category_active']),
        ('ohe_position', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['main_position_cat']),
        ('ohe_speciality', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['speciality_cat']),
        ('ohe_district', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['district']),
    ]
    transformers = [(n, t, c) for n, t, c in transformers if all(x in features for x in c)]
    return ColumnTransformer(transformers, remainder='drop', verbose_feature_names_out=False)


def build_preprocess_scaled(features):
    std_cols = [f for f in features if f in [
        'n_unique_subjects', 'n_unique_parallel', 'experience_age_pedagogic',
        'buildings_num', 'students_num', 'school_size', 'students_per_teacher',
        'pkpp_cum_courses', 'pkpp_cum_hours', 'pkpp_cum_types',
    ]]
    minmax_cols = [f for f in features if f in ['min_parallel', 'max_parallel']]
    robust_cols = [f for f in features if f in ['experience_age_other']]
    bin_cols = [f for f in features if f in ['sex', 'class_manager']]

    transformers = [
        ('te', TargetEncoder(target_type='binary', smooth='auto', cv=5, random_state=42), ['school_num']),
        ('std', StandardScaler(), std_cols),
        ('minmax', MinMaxScaler(), minmax_cols),
        ('robust', RobustScaler(), robust_cols),
        ('bin', 'passthrough', bin_cols),
        ('academic', academic_degree_step, ['academic_degree_cat']),
        ('qual', Pipeline([('map', category_qual_grouped_step), ('scaler', StandardScaler())]), ['category_qual_cat']),
        ('active', Pipeline([
            ('ord', OrdinalEncoder(categories=[['не было', 'не активно', 'активно']],
                                   handle_unknown='use_encoded_value', unknown_value=-1)),
            ('scaler', MinMaxScaler()),
        ]), ['category_active']),
        ('ohe_position', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['main_position_cat']),
        ('ohe_speciality', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['speciality_cat']),
        ('ohe_district', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['district']),
    ]
    transformers = [(n, t, c) for n, t, c in transformers if all(x in features for x in c)]
    return ColumnTransformer(transformers, remainder='drop', verbose_feature_names_out=False)

# --- 4. Train / test ---
train = df_pedagog_clean[df_pedagog_clean['year'] == 2022]
test  = df_pedagog_clean[df_pedagog_clean['year'] == 2023]
y_train = train['is_out_next_year'].values
y_test  = test['is_out_next_year'].values

print("Train:", train.shape, "Test:", test.shape)
print("Базовый рейт train:", f"{y_train.mean()*100:.2f}%", "test:", f"{y_test.mean()*100:.2f}%")

Train: (17370, 28) Test: (17337, 28)
Базовый рейт train: 11.42% test: 12.08%


In [66]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, fbeta_score, recall_score, balanced_accuracy_score, roc_auc_score,
    precision_recall_curve, auc,
)
import pandas as pd

# FEATURES_1..4 — уже определены
variants = [
    ('1_базовые', FEATURES_1),
    ('2_школы_и_sp', FEATURES_2),
    ('3_школы_без_sp', FEATURES_3),
    ('4_базовые_ПК', FEATURES_4),
]

train = df_pedagog_clean[df_pedagog_clean['year'] == 2022]
test = df_pedagog_clean[df_pedagog_clean['year'] == 2023]
y_train = train['is_out_next_year'].values
y_test = test['is_out_next_year'].values

rf_params = dict(n_estimators=300, max_depth=10,
                 min_samples_leaf=4, min_samples_split=10,  class_weight='balanced',
                 random_state=42, n_jobs=-1)
lr_params = dict(C=1.0, max_iter=1000, class_weight='balanced',
                 random_state=42, n_jobs=-1)

rows_rf, rows_lr = [], []
pipes_rf, pipes_lr = {}, {}

for name, feats in variants:
    # RF
    p = Pipeline([('prep', build_preprocess_tree(feats)),
                  ('clf', RandomForestClassifier(**rf_params))])
    p.fit(train[feats], y_train)
    yp = p.predict(test[feats]); ypr = p.predict_proba(test[feats])[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, ypr)
    rows_rf.append(dict(
        variant=name,
        f1=f1_score(y_test, yp), f2=fbeta_score(y_test, yp, beta=2),
        recall=recall_score(y_test, yp),
        balanced_acc=balanced_accuracy_score(y_test, yp),
        roc_auc=roc_auc_score(y_test, ypr),
        pr_auc=auc(rec, prec),
    ))
    pipes_rf[name] = p

    # LR
    p = Pipeline([('prep', build_preprocess_scaled(feats)),
                  ('clf', LogisticRegression(**lr_params))])
    p.fit(train[feats], y_train)
    yp = p.predict(test[feats]); ypr = p.predict_proba(test[feats])[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, ypr)
    rows_lr.append(dict(
        variant=name,
        f1=f1_score(y_test, yp), f2=fbeta_score(y_test, yp, beta=2),
        recall=recall_score(y_test, yp),
        balanced_acc=balanced_accuracy_score(y_test, yp),
        roc_auc=roc_auc_score(y_test, ypr),
        pr_auc=auc(rec, prec),
    ))
    pipes_lr[name] = p

df_rf = pd.DataFrame(rows_rf).set_index('variant')
df_lr = pd.DataFrame(rows_lr).set_index('variant')

print("=== RF ==="); print(df_rf.round(4))
print("\n=== LR ==="); print(df_lr.round(4))

=== RF ===
                    f1      f2  recall  balanced_acc  roc_auc  pr_auc
variant                                                              
1_базовые       0.3201  0.3689  0.4107        0.6260   0.7139  0.2504
2_школы_и_sp    0.2942  0.2942  0.2942        0.5986   0.7054  0.2590
3_школы_без_sp  0.2920  0.2910  0.2904        0.5972   0.7040  0.2572
4_базовые_ПК    0.3013  0.3013  0.3013        0.6026   0.7122  0.2582

=== LR ===
                    f1      f2  recall  balanced_acc  roc_auc  pr_auc
variant                                                              
1_базовые       0.3017  0.4304  0.6017        0.6368   0.6881  0.2452
2_школы_и_sp    0.3304  0.3825  0.4274        0.6340   0.6974  0.2615
3_школы_без_sp  0.3295  0.3827  0.4288        0.6338   0.6972  0.2614
4_базовые_ПК    0.3230  0.3783  0.4269        0.6299   0.6968  0.2597


In [68]:
def top_importance(pipe, k=10):
    names = pipe.named_steps['prep'].get_feature_names_out()
    imp = pipe.named_steps['clf'].feature_importances_
    return (pd.DataFrame({'feature': names, 'importance': imp})
            .sort_values('importance', ascending=False).head(k)
            .reset_index(drop=True))

for name in ['1_базовые', '2_школы_и_sp', '3_школы_без_sp', '4_базовые_ПК']:
    print(f"\n=== RF balanced: {name} — топ-10 ===")
    print(top_importance(pipes_rf[name]))


=== RF balanced: 1_базовые — топ-10 ===
                    feature  importance
0  experience_age_pedagogic    0.235647
1                school_num    0.156164
2             class_manager    0.101374
3      experience_age_other    0.085761
4         n_unique_subjects    0.069687
5              max_parallel    0.068976
6         n_unique_parallel    0.065556
7              min_parallel    0.059672
8     category_qual_grouped    0.055882
9           category_active    0.051879

=== RF balanced: 2_школы_и_sp — топ-10 ===
                    feature  importance
0            pkpp_cum_hours    0.126175
1  experience_age_pedagogic    0.120727
2          pkpp_cum_courses    0.104263
3                school_num    0.074767
4            pkpp_cum_types    0.063293
5      students_per_teacher    0.053600
6              students_num    0.051870
7               school_size    0.050729
8              max_parallel    0.038292
9             class_manager    0.037648

=== RF balanced: 3_школы_без_sp — 

In [67]:
def top_coefs(pipe, k=10):
    names = pipe.named_steps['prep'].get_feature_names_out()
    coefs = pipe.named_steps['clf'].coef_[0]
    return (pd.DataFrame({'feature': names, 'coef': coefs,
                          'abs_coef': np.abs(coefs)})
            .sort_values('abs_coef', ascending=False).head(k)
            [['feature', 'coef']].reset_index(drop=True))

for name in ['1_базовые', '2_школы_и_sp', '3_школы_без_sp', '4_базовые_ПК']:
    print(f"\n=== LR: {name} — топ-10 по |coef| ===")
    print(top_coefs(pipes_lr[name]))


=== LR: 1_базовые — топ-10 по |coef| ===
                          feature      coef
0                      school_num  3.545972
1                   class_manager -0.632844
2                    min_parallel  0.451084
3       main_position_cat_учитель -0.417480
4        main_position_cat_другой  0.279993
5           category_qual_grouped -0.212201
6      speciality_cat_воспитатель -0.202141
7             academic_degree_bin -0.197688
8                 category_active -0.195356
9  main_position_cat_руководитель  0.183663

=== LR: 2_школы_и_sp — топ-10 по |coef| ===
                      feature      coef
0                  school_num  3.388004
1            pkpp_cum_courses -0.548922
2                min_parallel  0.538623
3               class_manager -0.486477
4  speciality_cat_воспитатель -0.275982
5       category_qual_grouped -0.193459
6    experience_age_pedagogic -0.190076
7                 school_size -0.188555
8     district_Северный округ -0.177205
9   main_position_cat_учитель